# Day 16 — MLOps: Experiment Tracking, Model Registry & Reproducible Training (Layer 16)

The Production ML Engineering layer. It turns the platform from "scripts + `.joblib` files" into a
**reproducible ML system** — every run tracked as data, every model versioned with its lineage, every
artifact hash-referenced, one command to reproduce, and a registry that answers *"what's in production
and why?"* with metrics.

**This layer WRAPS the models the platform already trained** — the mandatory Logistic-Regression
baseline and the calibrated **LightGBM** incumbent (test PR-AUC **0.688**, threshold **0.329**). It does
**not** retrain them away or overwrite them.

**The honesty discipline (as every day):** we wrap the models that *exist* and add honest XGBoost /
RandomForest comparison candidates on the same real split. We **defer** — never fabricate — a neural
network the platform never trained, Bayesian/optuna tuning (no dependency), a remote MLflow server, and
drift/continuous-training. This layer is *faiss-free* (LightGBM/XGBoost/sklearn only), so no OpenMP guard.

> This notebook runs in an **isolated sandbox** (`artifacts/_notebook16/`) so executing it never mutates
> the platform's canonical registry or artifact store.

In [1]:
import sys, os, warnings, logging, json, copy
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
from src.utils.config import load_config

# ISOLATED config — point every Day-16 output at a sandbox so the canonical registry is untouched.
cfg = copy.deepcopy(load_config())
NB = "artifacts/_notebook16"
cfg["day16"]["registry"]["path"] = f"{NB}/model_registry.json"
cfg["day16"]["artifacts"].update(index_path=f"{NB}/artifact_index.json", store_root=f"{NB}/store")
cfg["day16"]["training"]["runs_dir"] = f"{NB}/training_runs"
cfg["day16"]["selection"]["out_path"] = f"{NB}/model_selection.json"
cfg["day16"]["mlflow"].update(tracking_uri=f"sqlite:///{NB}/mlflow.db",
                              artifact_location=f"./{NB}/mlartifacts", experiment_name="avip-notebook16")
print("Isolated MLOps sandbox:", NB)

Isolated MLOps sandbox: artifacts/_notebook16


## 1 — MLflow: a local, serverless backend

Experiment tracking needs a place to put runs. Not a server — a **local** backend. (MLflow 3.15
deprecated the plain file store, so we use a local **SQLite** backend + a local artifact dir; still one
file, no service. `mlflow ui` is an optional viewer.)

In [2]:
from src.mlops.mlflow_setup import MLflowSetup
print(MLflowSetup(cfg).describe())

{'tracking_uri': 'sqlite:////Users/Yash/AI-Ventutre-Intelligence-Platform/artifacts/_notebook16/mlflow.db', 'artifact_uri': 'file:///Users/Yash/AI-Ventutre-Intelligence-Platform/artifacts/_notebook16/mlartifacts', 'experiment_name': 'avip-notebook16', 'experiment_id': '1'}


## 2 — The reproducibility fingerprint

A metric is only trustworthy if you can **re-derive** it. So every run pins the six dimensions that
shaped it — dataset hash, split hash, seed, git commit (+ dirty flag), and library versions — plus a
deterministic version string. `dirty: True` is the fingerprint honestly telling you this tree has
uncommitted work.

In [3]:
from src.mlops.version_manager import VersionManager
vm = VersionManager(cfg); fp = vm.fingerprint()
print("version_string :", vm.version_string(fp))
print("dataset sha    :", fp["dataset"]["sha256"][:12])
print("split (train)  :", fp["split"]["train_sha256"][:12])
print("git            :", fp["git"])
print("libraries      :", {k: fp["libraries"][k] for k in ("python", "lightgbm", "sklearn", "xgboost")})

version_string : 7711b9fc22ae
dataset sha    : 90bf582925cd
split (train)  : 99f0a3446085
git            : {'commit': '3ea7e1db0bb6f0cdf21aef7e6b6ccb010feb7ef9', 'dirty': True}
libraries      : {'python': '3.14.2', 'lightgbm': '4.7.0', 'sklearn': '1.9.0', 'xgboost': '3.3.0'}


## 3 — Reproducible training (the centerpiece)

One command: load → train (+calibrate) → evaluate → save **versioned** → track → register. Two frozen
guarantees: it **reproduces** the incumbent (re-training from config lands back on ≈ 0.688 PR-AUC), and
it **never overwrites** the incumbent (outputs go to a versioned run dir; the new model is registered as
*Development*, not promoted).

*(Calibration retrains LightGBM several times — this cell takes ~1–2 minutes.)*

In [4]:
from src.training.train_pipeline import TrainingPipeline
rep = TrainingPipeline(cfg).run()
print("run version         :", rep["version"])
print("test PR-AUC         :", rep["test_metrics"]["pr_auc"])
print("incumbent PR-AUC    :", rep["incumbent_pr_auc"])
print("reproduces incumbent:", rep["reproduces_incumbent"])
print("test F1 / recall    :", rep["test_metrics"]["f1"], "/", rep["test_metrics"]["recall"],
      "@thr", rep["threshold"])

run version         : lightgbm-7711b9fc22ae
test PR-AUC         : 0.6883
incumbent PR-AUC    : 0.6884
reproduces incumbent: True
test F1 / recall    : 0.6664 / 0.8131 @thr 0.329


## 4 — The model registry: lifecycle + single-Production invariant

The registry is the source of truth for "what's deployed and why." Statuses move
`Development → Staging → Production → Archived`, and there can be **at most one Production version per
slot** — promoting a version auto-archives the incumbent. Here we register the two real curated models
(baseline + incumbent) alongside the candidate the training pipeline just added.

In [5]:
from src.mlops.model_registry import ModelRegistry
reg = ModelRegistry(cfg); SLOT = "funding_success_classifier"
meta = json.load(open("artifacts/models/model_metadata.json"))
cal = meta["validation_metrics"]["calibrated_val"]
logreg = json.load(open("artifacts/models/logreg_results.json"))["logreg_l2"]["metrics"]

reg.register(name=SLOT, version="logreg-l2-baseline", model_type="LogisticRegression(l2)",
             metrics=logreg, role="baseline", status="Archived")
reg.register(name=SLOT, version="lightgbm-1.0.0", model_type=meta["model"]["type"],
             metrics={**cal, "threshold": meta["decision_threshold"]}, role="incumbent", status="Staging")
reg.promote(SLOT, "lightgbm-1.0.0", "Production")

for e in reg.list(name=SLOT):
    print(f"  {e['version']:<24} {e['status']:<12} role={e['role']:<10} pr_auc={e['metrics'].get('pr_auc')}")
print("\nProduction:", reg.production(SLOT)["version"], "| #Production entries:", len(reg.list(status="Production")))

  lightgbm-7711b9fc22ae    Development  role=candidate  pr_auc=0.6883
  logreg-l2-baseline       Archived     role=baseline   pr_auc=0.6264
  lightgbm-1.0.0           Production   role=incumbent  pr_auc=0.6884

Production: lightgbm-1.0.0 | #Production entries: 1


## 5 — Artifact versioning: copy small, reference large

Versioning artifacts isn't *copying* artifacts. Small files are copied into a managed store; the
**407 MB** FAISS index and embeddings are **referenced by content hash** — pointed at, never duplicated.

In [6]:
from src.mlops.artifact_manager import ArtifactManager
am = ArtifactManager(cfg)
am.register("artifacts/models/calibrated_model.joblib", "models")           # small -> copied
am.register("artifacts/vector_db/faiss.index", "vectorizers")               # 407MB -> referenced
am.register("artifacts/embeddings/startup_embeddings.npy", "vectorizers", name="embeddings.npy")

for e in am.list():
    print(f"  {e['category']:<11} {e['name']:<26} {e['mode']:<11} {e['size_bytes']/(1<<20):7.1f} MB")
d = am.describe()
print(f"\n{d['by_mode']}  ->  {d['mb_saved_by_reference']} MB saved by referencing instead of copying")

  models      lightgbm-7711b9fc22ae.joblib copied         20.5 MB
  vectorizers faiss.index                referenced    388.4 MB
  models      calibrated_model.joblib    copied         20.5 MB
  vectorizers embeddings.npy             referenced    388.4 MB

{'copied': 2, 'referenced': 2}  ->  776.9 MB saved by referencing instead of copying


## 6 — Automated selection: fair comparison, gated promotion

Two honesty rules. **Fair F1:** each model is scored at *its own* F1-optimal threshold — otherwise the
calibrated incumbent (0.329) would be compared against challengers stuck at 0.5. **Gated promotion:**
`apply()` promotes only if a challenger beats the incumbent by a real margin — otherwise it's a no-op
and we **keep the incumbent**.

*(Trains the incumbent + baseline + an XGBoost challenger — ~2 minutes. RandomForest is omitted here
for time; the reference records it at **1023 MB / 13 ms** — a vivid footprint-disqualification even
when its F1 is close.)*

In [7]:
from src.training.model_selector import ModelSelector
sel = ModelSelector(cfg)
dec = sel.run(families=["xgboost"])           # incumbent (calibrated) + baseline (logreg) + xgboost

print("challengers (F1 at own optimal threshold):")
for c in dec["ranked"]:
    m, f = c["metrics"], c["footprint"]
    print(f"  {c['name']:<12} F1={m['f1']:.4f} pr_auc={m['pr_auc']:.4f} thr={c['threshold']} "
          f"| {f['size_mb']}MB {f['latency_ms_per_1k']}ms/1k")
print(f"\nincumbent lightgbm F1={dec['incumbent']['metrics']['f1']:.4f} "
      f"pr_auc={dec['incumbent']['metrics']['pr_auc']:.4f}")
print(f"baseline  logreg   F1={dec['baseline']['metrics']['f1']:.4f}  (floor)")
print(f"\nRECOMMENDATION: {dec['recommendation']}")
print("RATIONALE:", dec["rationale"])
print("apply (gated):", sel.apply(dec))

challengers (F1 at own optimal threshold):
  xgboost      F1=0.6576 pr_auc=0.6707 thr=0.34 | 0.36MB 0.416ms/1k

incumbent lightgbm F1=0.6664 pr_auc=0.6883
baseline  logreg   F1=0.6346  (floor)

RECOMMENDATION: keep_incumbent
RATIONALE: Best challenger xgboost F1=0.6576 vs incumbent 0.6664 (margin required 0.005). Does NOT beat the incumbent by the required margin → keep incumbent.
apply (gated): {'promoted': False, 'reason': 'recommendation is keep_incumbent'}


## Summary — the pattern, and the honesty

**Layer 16 = MLOps.** Every run is tracked with a reproducibility fingerprint, so re-training lands back
on 0.688 — verified, not assumed. Models live in a registry with a Development-to-Production lifecycle
and a single-Production invariant; artifacts are versioned by content hash (a 407 MB index referenced,
not copied); and selection is **fair** (each model at its own threshold) and **gated** (promote only on a
real margin) — which honestly recommended **keeping the incumbent**. The whole day resists scope creep:
wrap the models that exist, defer the ones that don't.

> **Interview soundbite.** *"Day 16 makes the platform reproducible without touching the production
> model. Every run carries a fingerprint — dataset hash, split, seed, git SHA, library versions — so
> re-training reproduces 0.688 PR-AUC, and I verify by hash that the incumbent's files never change.
> Models live in a registry with a single-Production invariant, artifacts are versioned by content hash
> so I reference a 407 MB index instead of copying it, and selection is fair and gated — each model at
> its own optimal threshold, promotion only on a real margin — which honestly told me to keep the
> incumbent. The discipline is scope: I wrapped the models I actually trained and deferred a neural
> network I never built."*